In [1]:
import pandas as pd
import numpy as np

import re

import torch
from catboost import CatBoostRegressor, Pool
from transformers import AutoModel, AutoTokenizer
from wordfreq import zipf_frequency

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

/home/mihnea/Programming/Nitro NLP/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

In [3]:
TRANSFORMER_MODEL = "dumitrescustefan/bert-base-romanian-cased-v1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONTEXT_RADIUS = 24
MAX_LENGTH = 128
CATEGORICAL_FEATURES = ["participant_id", "text"]
ATTENTION_CACHE = {}

tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL, use_fast=True)
attention_model = AutoModel.from_pretrained(TRANSFORMER_MODEL, attn_implementation="eager").to(DEVICE)
attention_model.eval()


def eval_metric(y_true, preds):
    y_true = y_true.astype(float)
    preds = preds.astype(float)
    r2 = r2_score(y_true, preds, sample_weight=None, force_finite=True)
    r2 = max(0, r2)
    pears = pearsonr(y_true, preds)[0]
    if np.isnan(pears):
        pears = 0.0
    pears = np.abs(pears)
    return 100 * (pears + r2) / 2


def word_position(word_id):
    match = re.search(r"_(\d+)$", str(word_id))
    return int(match.group(1)) if match else -1


def build_text_index(df):
    unique_words = df[["text", "word_id", "word"]].drop_duplicates("word_id").copy()
    unique_words["position"] = unique_words["word_id"].map(word_position)
    text_index = {}

    for text, text_df in unique_words.sort_values(["text", "position"]).groupby("text"):
        ids = text_df["word_id"].tolist()
        words = text_df["word"].fillna("").astype(str).tolist()
        text_index[text] = {
            "ids": ids,
            "words": words,
            "position_by_id": {word_id: i for i, word_id in enumerate(ids)},
        }

    return text_index


def get_attention_features(text_index, text, word_id):
    item = text_index.get(text)
    if item is None or word_id not in item["position_by_id"]:
        return np.zeros(6, dtype=float)

    center = item["position_by_id"][word_id]
    start = max(0, center - CONTEXT_RADIUS)
    end = min(len(item["words"]), center + CONTEXT_RADIUS + 1)
    local_words = item["words"][start:end]
    target_word_idx = center - start

    encoded = tokenizer(
        local_words,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    word_ids = encoded.word_ids(0)
    target_tokens = [i for i, idx in enumerate(word_ids) if idx == target_word_idx]
    if not target_tokens:
        return np.zeros(6, dtype=float)

    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
    with torch.inference_mode():
        outputs = attention_model(**encoded, output_attentions=True)

    attention = outputs.attentions[-1][0].detach().float().cpu().numpy()
    target_attention = attention[:, target_tokens, :]
    target_attention_mean = target_attention.mean(axis=(0, 1))
    entropy = -(target_attention_mean * np.log(target_attention_mean + 1e-12)).sum()

    return np.array([
        attention[:, 0, target_tokens].mean(),
        attention[:, target_tokens, 0].mean(),
        attention[:, :, target_tokens].mean(),
        attention[:, :, target_tokens].max(),
        entropy,
        len(target_tokens),
    ], dtype=float)


def featurize(df, context_df=None):
    if context_df is None:
        context_df = df

    result = pd.DataFrame(index=df.index)
    words = df["word"].fillna("").astype(str)

    result["word_len"] = words.map(len)
    result["zipf_ro"] = words.map(lambda word: zipf_frequency(word, "ro"))
    result["unique_chars"] = words.map(lambda word: len(set(word)))
    result["is_lower"] = words.map(str.islower).astype(int)
    result["is_title"] = words.map(str.istitle).astype(int)
    result["has_digit"] = words.map(lambda word: any(ch.isdigit() for ch in word)).astype(int)
    result["has_punct"] = words.map(lambda word: any(not ch.isalnum() for ch in word)).astype(int)
    result["position"] = df["word_id"].map(word_position)
    result["participant_id"] = df["participant_id"].astype(str)
    result["text"] = df["text"].astype(str)

    text_index = build_text_index(context_df)
    attention_rows = []
    for text, word_id in zip(df["text"].astype(str), df["word_id"].astype(str)):
        cache_key = (text, word_id)
        if cache_key not in ATTENTION_CACHE:
            ATTENTION_CACHE[cache_key] = get_attention_features(text_index, text, word_id)
        attention_rows.append(ATTENTION_CACHE[cache_key])

    attention_df = pd.DataFrame(
        attention_rows,
        index=df.index,
        columns=[
            "attn_cls_to_word",
            "attn_word_to_cls",
            "attn_received_mean",
            "attn_received_max",
            "attn_entropy",
            "subtoken_count",
        ],
    )

    return pd.concat([result, attention_df], axis=1)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12083.66it/s]
[transformers] BertModel LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
class HackathonMetric:
    def is_max_optimal(self):
        return True

    def evaluate(self, approxes, target, weight):
        preds = np.asarray(approxes[0], dtype=float)
        y_true = np.asarray(target, dtype=float)
        score = eval_metric(y_true, np.clip(preds, 0, None))
        return score, 1.0

    def get_final_error(self, error, weight):
        return error


def make_model():
    return CatBoostRegressor(
        loss_function="RMSE",
        eval_metric=HackathonMetric(),
        iterations=1200,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=6.0,
        random_seed=42,
        od_type="Iter",
        od_wait=80,
        verbose=False,
        allow_writing_files=False,
    )

In [5]:
all_data = pd.concat([df_train.drop(columns=["answer"]), df_test], ignore_index=True)
X = featurize(df_train, all_data)
X_test = featurize(df_test, all_data)
y = df_train["answer"].to_numpy()

In [6]:
n_bins = 10

y_bins = np.digitize(
    y,
    np.quantile(y, np.linspace(0, 1, n_bins + 1)[1:-1])
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [7]:
oof = np.zeros(len(y))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_bins), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    train_pool = Pool(X_tr, y_tr, cat_features=CATEGORICAL_FEATURES)
    val_pool = Pool(X_val, y_val, cat_features=CATEGORICAL_FEATURES)
    test_pool = Pool(X_test, cat_features=CATEGORICAL_FEATURES)

    model = make_model()
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    val_preds = model.predict(val_pool)
    val_preds = np.clip(val_preds, 0, None)

    oof[val_idx] = val_preds

    test_fold_preds = model.predict(test_pool)
    test_fold_preds = np.clip(test_fold_preds, 0, None)

    test_preds += test_fold_preds / skf.n_splits

    print(f"Fold {fold}: {eval_metric(y_val, val_preds):.4f}")

print("OOF:", eval_metric(y, oof))

/home/mihnea/Programming/Nitro NLP/.venv/lib/python3.11/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


Fold 1: 52.4494


/home/mihnea/Programming/Nitro NLP/.venv/lib/python3.11/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


Fold 2: 53.2837


/home/mihnea/Programming/Nitro NLP/.venv/lib/python3.11/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


Fold 3: 51.8527


/home/mihnea/Programming/Nitro NLP/.venv/lib/python3.11/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


Fold 4: 52.6023


/home/mihnea/Programming/Nitro NLP/.venv/lib/python3.11/site-packages/catboost/core.py:2627: UserWarning: Failed to import numba for optimizing custom metrics and objectives
  _check_train_params(params)


Fold 5: 51.8972
OOF: 52.405819319662726


In [9]:
submission = pd.read_csv("sample_output.csv")
submission["answer"] = test_preds
submission.to_csv("submission.csv", index=False)